# Block the Bot · 防守者 LoRA 微调（Qwen3-0.6B）

右边 **Settings**：Accelerator 选 **GPU T4 x2**（或 P100），**Internet on**。
**Add Input** 加上数据集 `block-the-bot-sft`（里面是 `train.jsonl`、`val.jsonl`、`train_lora.py`）。然后 **Run All**，或 **Save Version → Save & Run All** 在后台跑。

跑完在 **Output** 里下载 `block-the-bot-lora.zip`，解压到本机项目的 `training/out/`，再运行 `scripts/convert_lora.sh` 转成 GGUF。

In [ ]:
!pip install -q transformers==4.51.3 peft==0.15.2 accelerate==1.6.0
import torch, glob, os
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '没有 GPU：右边 Settings 里选 Accelerator')

In [ ]:
# 找到上传的数据集
def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'没找到 {name}：右边 Add Input 加上 block-the-bot-sft'
    return hits[0]
TRAIN, VAL, SCRIPT = find('train.jsonl'), find('val.jsonl'), find('train_lora.py')
print(TRAIN, VAL, SCRIPT)

In [ ]:
# 底模：Hugging Face 上的 Qwen/Qwen3-0.6B（Apache-2.0，约 1.5 GB；需要开 Internet）
from huggingface_hub import snapshot_download
BASE = snapshot_download('Qwen/Qwen3-0.6B', local_dir='/kaggle/working/base',
                         allow_patterns=['*.json', '*.safetensors', '*.txt', 'merges.txt', 'vocab.json', 'LICENSE'])
print(BASE)

In [ ]:
# 训练：只用 L2（Top-8）样本，跑两遍。L1 样本平均 749 token（最长 1524），以后要做消融时再单独训
# T4 没有原生 bfloat16，脚本会自动用 float16 混合精度
!python {SCRIPT} --data {TRAIN} --val-data {VAL} --base {BASE} --out /kaggle/working/out --level L2 --epochs 2 --micro 8 --batch 16 --max-len 512 --check 200

In [ ]:
# 打包适配器（在右边 Output 里下载）
!cd /kaggle/working/out && zip -qr /kaggle/working/block-the-bot-lora.zip adapter train_args.json && ls -la /kaggle/working/block-the-bot-lora.zip